## Silver Cleaning 

## Necessary Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Read Bronze Table Before Cleaning

In [0]:
# Read bronze table
df_bronze = spark.read.table("digital_banking.bronze.bronze_customers")

print(f"Bronze records count: {df_bronze.count()}")
display(df_bronze.limit(10))

## Find Col. Having Data-Type `String` To Remove Leading & Trailing Spaces

In [0]:
# Step 1: Trim leading/trailing spaces from all string columns
string_columns = [field.name for field in df_bronze.schema.fields if str(field.dataType) == "StringType()"]
print(string_columns)

df_cleaned = df_bronze
for col in string_columns:
    df_cleaned = df_cleaned.withColumn(col, F.trim(F.col(col)))

## Removing Invalid Email Addresses

In [0]:
# Step 2: Validate and clean email addresses
# Email regex pattern: basic validation
email_pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

df_cleaned = df_cleaned.withColumn(
    "email_valid",
    F.when(
        (F.col("email").isNotNull()) & 
        (F.col("email").rlike(email_pattern)),
        F.col("email")
    ).otherwise(None)
)

## Cast Col. To `DateType()` 

In [0]:
# Step 3: Validate and convert date fields
# Handle date_of_birth
df_cleaned = df_cleaned.withColumn(
    "date_of_birth_clean",
    F.when(
        F.col("date_of_birth").isNotNull(),
        F.to_date(F.col("date_of_birth"), "yyyy-MM-dd")
    ).otherwise(None)
)

# Handle registration_date
df_cleaned = df_cleaned.withColumn(
    "registration_date_clean",
    F.when(
        F.col("registration_date").isNotNull(),
        F.to_date(F.col("registration_date"), "yyyy-MM-dd")
    ).otherwise(None)
)

# Handle updated_at
df_cleaned = df_cleaned.withColumn(
    "updated_at_clean",
    F.when(
        F.col("updated_at").isNotNull(),
        F.to_date(F.col("updated_at"), "yyyy-MM-dd")
    ).otherwise(None)
)


## Remove Future Birth Dates

### Realistic Age Constraint
If someone was born on `January 1, 1900`, they would be `126` years old today (September 3, 2026).

The oldest verified human lifespan is around 122 years
Anyone born before 1900 would be 126+ years old - extremely unlikely
This catches obvious data entry errors like birth years in the 1800s

In [0]:
# Step 4: Additional date validations - reject future birth dates and unreasonable dates
df_cleaned = df_cleaned.withColumn(
    "date_of_birth_clean",
    F.when(
        (F.col("date_of_birth_clean").isNotNull()) &
        (F.col("date_of_birth_clean") <= F.current_date()) &
        (F.col("date_of_birth_clean") >= F.lit("1900-01-01")),
        F.col("date_of_birth_clean")
    ).otherwise(None)
)

## Remove Duplicate Customer ID Using Window Function 
> Keeping The Latest One's In Silver Cleaned Table ie.(**silver_customers**)
> Preserving The Duplicated Customer ID's in `df_duplicates` **dataframe**

In [0]:
# Step 5: Remove duplicate records based on customer_id (keep the most recent updated_at)
window_spec = Window.partitionBy("customer_id").orderBy(F.col("updated_at_clean").desc_nulls_last())
df_cleaned = df_cleaned.withColumn("row_num", F.row_number().over(window_spec))

# Separate duplicates from valid records
df_deduped = df_cleaned.filter(F.col("row_num") == 1).drop("row_num")  # Keep most recent (row_num = 1)
df_duplicates = df_cleaned.filter(F.col("row_num") > 1)  # Save duplicates (row_num > 1)

print(f"Valid records (most recent): {df_deduped.count()}")
print(f"Duplicate records to save: {df_duplicates.count()}")

# Step 6: Handle missing critical values - filter out records with missing customer_id
df_final = df_deduped.filter(F.col("customer_id").isNotNull())

In [0]:

# Step 7: Create final silver table with cleaned columns and proper types
df_silver = df_final.select(
    F.col("customer_id"),
    F.col("first_name"),
    F.col("last_name"),
    F.col("date_of_birth_clean").alias("date_of_birth"),
    F.col("email_valid").alias("email"),
    F.col("phone"),
    F.col("address"),
    F.col("city"),
    F.col("state"),
    F.col("postal_code"),
    F.col("customer_segment"),
    F.col("customer_status"),
    F.col("registration_date_clean").alias("registration_date"),
    F.col("updated_at_clean").alias("updated_at")
)

In [0]:
print(f"Silver records count after cleaning: {df_silver.count()}")

# Show data quality summary
print(f"Records removed (duplicates + invalid): {df_bronze.count() - df_silver.count()}")
print(f"Invalid emails found: {df_bronze.count() - df_final.filter(F.col('email_valid').isNotNull()).count()}")
print(f"Invalid dates of birth: {df_bronze.count() - df_final.filter(F.col('date_of_birth_clean').isNotNull()).count()}")

# Step 8: Write to silver table
df_silver.write.mode("overwrite").saveAsTable("digital_banking.silver.silver_customers")

# Display sample of cleaned data
display(df_silver.limit(10))